Recommend only using Pyvis to create graphs for single CDRs or heavily filtered data from across multiple CDRs.

In [122]:
from pyvis.network import Network
import pandas as pd

DIR = '../outputs_sample_1/notional_call_detail_records/'
FILENAME = 'PICKUP_DRIVER001_CDR.csv'
OUTPUT_NAME = f'Pyvis_Graph_{FILENAME[:-4]}.html'

In [133]:
df = pd.read_csv(DIR+FILENAME)
print(df.shape)
df.head(3).T

(6198, 17)


,0,1,2
RecordID,7B8AFLDPBH,LR0N0NH0B0,5TJ7NZKYRZ
SubscriberID,PICKUP_DRIVER001,PICKUP_DRIVER001,PICKUP_DRIVER001
SubscriberRole,US_PICKUP_DRIVER,US_PICKUP_DRIVER,US_PICKUP_DRIVER
EventTimestamp,2026-01-25 02:32:47,2026-05-24 20:06:30,2026-06-03 00:11:15
Direction,MT,MT,MT
EventType,VOICE,SMS,SMS
CallingNumber,12137093109,12137093109,12137093109
CalledNumber,13108123528,13108123528,13108123528
DurationSec,283.0,NaN,NaN
SMSLength,NaN,123.0,29.0


In [134]:
# Groupby on calling, called, and event type to get counts for each event
grouped_events = df.groupby(['CallingNumber', 'CalledNumber', 'EventType'])['RecordID'].count().reset_index()

# Create col to concat event types and counts
grouped_events['EventTypeCount'] = grouped_events.apply(lambda x: f"{x['RecordID']} {x['EventType']}", axis=1)

# Flatten events into 1 edge per pair and aggregate event types and counts
agg_events = grouped_events.groupby(['CallingNumber', 'CalledNumber']).agg({
        'EventType': ', '.join,
        'RecordID': 'sum',
        'EventTypeCount': ', '.join
    }).reset_index()

agg_events.rename(mapper={
    'RecordID': 'TotalEventCount',
    'EventTypeCount': 'EventCountByType'
}, axis=1, inplace=True)

agg_events.head()

,CallingNumber,CalledNumber,EventType,TotalEventCount,EventCountByType
0,12135727747,13108123528,"SMS, VOICE",112,"46 SMS, 66 VOICE"
1,12137093109,13108123528,"SMS, VOICE",16,"6 SMS, 10 VOICE"
2,13104167888,13108123528,"SMS, VOICE",237,"93 SMS, 144 VOICE"
3,13105741375,13108123528,"SMS, VOICE",287,"103 SMS, 184 VOICE"
4,13105859458,13108123528,"SMS, VOICE",112,"52 SMS, 60 VOICE"


In [135]:
net = Network(
    width='100%',
    height='750px',
    directed=True,
    bgcolor='white',
    font_color='black'
    )

In [136]:
all_nodes = list(set(
    list(agg_events.CallingNumber.unique())
    + list(agg_events.CalledNumber.unique())
))
all_nodes = [int(i) for i in all_nodes]
all_nodes[0:3]

[13105859458, 12135727747, 13232467714]

In [137]:
all_edges = list(agg_events.itertuples(index=False, name=None))
all_edges[0:3]

[(12135727747, 13108123528, 'SMS, VOICE', 112, '46 SMS, 66 VOICE'),
 (12137093109, 13108123528, 'SMS, VOICE', 16, '6 SMS, 10 VOICE'),
 (13104167888, 13108123528, 'SMS, VOICE', 237, '93 SMS, 144 VOICE')]

In [138]:
# Add nodes ndividually 
# Allows more control over styling vs net.add_nodes()
for n in all_nodes:
    net.add_node(
        n_id=n,
        label=str(n), 
        title=str(n),
        shape='image', 
        image='demo_assets/phone_circle.png',
        size=10,
        physics=False
    )

print(len(net.get_nodes()))

44


In [139]:
# Add edges individually
# Allows more control over styling vs net.add_edges()
for e in all_edges:
    net.add_edge(
        source=e[0],
        to=e[1],
        # value=e[3], # makes graph messy
        label=str(e[3]),
        title=str(e[4]),
        color={
            'color': 'grey',
            'highlight': 'red',
            'inherit': False,
        },
        physics=False
    )

print(len(net.get_edges()))

62


In [140]:
net.set_edge_smooth('straightCross')
net.save_graph(OUTPUT_NAME)
print('done')

done
